### Parameters

In [0]:
# # catalog
# catalog = "workspace"

# # key column
# key_cols = "['flight_id']"
# key_cols_list = eval(key_cols)

# # cdc column
# cdc_col = "modifiedDate"

# # backdated refresh
# backdated_refresh = ""

# # source object
# source_object = "silver_flights"

# # source schema
# source_schema = "silver"

# # target schema
# target_schema = "gold"

# # target object
# target_object = "dimflights"

# # surrogate key name
# surrogate_key = "dimflightskey"

In [0]:
# '''
# This is the gold_dim builder notebook which is dynamic and able to convert any source into dimension table.
# we just need to pass the parameter which we will make dynamic also but before that lets try out to create it for dim_airports by changind some of the parameters
# '''
# # catalog
# catalog = "workspace"

# # key column
# key_cols = "['airport_id']"
# key_cols_list = eval(key_cols)

# # cdc column
# cdc_col = "modifiedDate"

# # backdated refresh
# backdated_refresh = ""

# # source object
# source_object = "silver_airports"

# # source schema
# source_schema = "silver"

# # target schema
# target_schema = "gold"

# # target object
# target_object = "dimairports"

# # surrogate key name
# surrogate_key = "dimairportskey"

In [0]:
'''
This is the gold_dim builder notebook which is dynamic and able to convert any source into dimension table.
we just need to pass the parameter which we will make dynamic also but before that lets try out to create it for dim_airports by changind some of the parameters
'''
# catalog
catalog = "workspace"

# key column
key_cols = "['passenger_id']"
key_cols_list = eval(key_cols)

# cdc column
cdc_col = "modifiedDate"

# backdated refresh
backdated_refresh = ""

# source object
source_object = "silver_passengers"

# source schema
source_schema = "silver"

# target schema
target_schema = "gold"

# target object
target_object = "dimpassengers"

# surrogate key name
surrogate_key = "dimpassengerskey"

### Incremental Data Ingestion

In [0]:
# no backdated refresh
if len(backdated_refresh) == 0:
    
    # if table exists in the destination
    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

        last_load = spark.sql(f"SELECT max({cdc_col})FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]
    else:

        last_load = "1900-01-01 00:00:00"
# yes the backdated_refresh
else:
    last_load = backdated_refresh

# test the load date
last_load

In [0]:
df_src = spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col} > '{last_load}'")

In [0]:
'''
we want a surrogate key to identitfy which are the old records and which are new ones.
now we perform the join between source and destination which is easy when we have table in the destination but it is lill bit  different when we are doing initial load because there will be no table.
'''

### old vs new records

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    # key column string for incremental load
    key_cols_string_incremental = ", ".join(key_cols_list)
    df_trg = spark.sql(f"""select {key_cols_string_incremental},{surrogate_key},create_date,update_date from {catalog}.{target_schema}.{target_object}""")

else:
    
    # key column string for initial load
    key_cols_string_init = [f" '' as {i}" for i in key_cols_list ]
    key_cols_string_init = ', '.join(key_cols_string_init)
    df_trg = spark.sql(f"""select {key_cols_string_init}, cast('0' as int) as {surrogate_key}, cast('1900-01-01 00:00:00' as timestamp) as create_date, cast('1900-01-01 00:00:00' as timestamp) as update_date where 1=0""")

In [0]:
display(df_trg)

#### join condition

In [0]:
join_condition = ' AND '.join([f"src.{i} = trg.{i}" for i in key_cols_list])

In [0]:
df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")

df_join = spark.sql(f"""
                     select src.*,
                            trg.{surrogate_key},
                            trg.create_date,
                            trg.update_date
                     from src
                     left join trg
                     on {join_condition}
                     """
                   )

In [0]:
display(df_join)

In [0]:
from pyspark.sql.functions import *

# old records
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())
# new records
df_new = df_join.filter(col(f'{surrogate_key}').isNull())

In [0]:
display(df_old)
display(df_new)

### Enriching the df's

#### preparing df_old

In [0]:
''' the thing is we want all the column from the source and surrogate key, create date and update date from the target table. and the create date will be the first time when the record was created and update date will be the refreshed date when the records get updated/refreshed after doing incremental load or if it is an initial load then the create date and update date will be same. '''

In [0]:
df_old_enr = df_old.withColumn('update_date', current_timestamp())

#### preparing df_new

In [0]:
# if table is there
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    max_surrogate_key = spark.sql(f"""
                                  select max({surrogate_key}) from {catalog}.{target_schema}.{target_object}
                                  """).collect()[0][0]
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+ monotonically_increasing_id())\
                       .withColumn('create_date', current_timestamp())\
                       .withColumn('update_date', current_timestamp())
    

# if table doesn't exist
else:
    max_surrogate_key = 0
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+ monotonically_increasing_id())\
                       .withColumn('create_date', current_timestamp())\
                       .withColumn('update_date', current_timestamp())

#### applying union between new vs old records

In [0]:
df_union = df_old_enr.unionByName(df_new_enr)

In [0]:
display(df_union)

### upsert

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_union.alias("src"), f"trg.{surrogate_key} = src.{surrogate_key}")\
            .whenMatchedUpdateAll(condition = f'src.{cdc_col} >= trg.{cdc_col}')\
            .whenNotMatchedInsertAll()\
            .execute()

else:
    df_union.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

In [0]:
%sql
select * from workspace.gold.dimpassengers